# M3: Entrenamiento de Keypoints de Ruedas / Ejes (YOLO11s-pose)
Un keypoint `centro` por cada rueda visible desde el lateral para estimación de ejes vehiculares.

Puedes preparar el dataset de dos formas:
- **Opción A (Recomendada y más rápida):** Auto-etiquetar automáticamente tus imágenes/video usando IA (YOLO-World) directamente en Colab, sin tener que hacer clicks manuales.
- **Opción B:** Descargar un dataset ya etiquetado desde Roboflow con tu API Key.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos /content/drive/MyDrive/peaje/runs
!git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje 2>/dev/null || git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r entrenamiento/requirements.txt


## Opción A (Recomendada): Auto-etiquetado con IA (Sin clicks manuales)
Sube tus imágenes (o un archivo `.zip` con tus 174 imágenes) arrastrándolo al panel izquierdo de Colab (`/content/`).
Esta celda usa YOLO-World para detectar todas las ruedas automáticamente y calcular el keypoint central de cada una.

In [ ]:
# ==============================================================================
# OPCIÓN A: AUTO-ETIQUETADO DE KEYPOINTS DE RUEDAS CON YOLO-WORLD
# ==============================================================================
import os, glob, shutil, random, zipfile
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Descomprimir zips subidos si existen
FOTOS_DIR = '/content/fotos_ruedas'
os.makedirs(FOTOS_DIR, exist_ok=True)
zip_subidos = [z for z in glob.glob('/content/*.zip') if 'dataset_ruedas_pose' not in z]
for z in zip_subidos:
    print(f'Descomprimiendo {z}...')
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall(FOTOS_DIR)

# Buscar todas las imágenes disponibles
todas_fotos = []
for root, _, files in os.walk(FOTOS_DIR):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            todas_fotos.append(os.path.join(root, f))

# También buscar fotos sueltas en /content/
fotos_sueltas = [f for f in glob.glob('/content/*.jpg') + glob.glob('/content/*.png') if not f.startswith('/content/sample')]
todas_fotos = sorted(list(set(todas_fotos + fotos_sueltas)))

print(f'Total de imágenes encontradas para auto-etiquetar: {len(todas_fotos)}')
if len(todas_fotos) == 0:
    print('⚠️ ATENCIÓN: Sube tus imágenes o archivo .zip al panel de archivos de Colab (/content/) y vuelve a correr esta celda.')
else:
    print('Cargando detector YOLO-World para localización precisa de ruedas...')
    world_model = YOLO('yolov8m-worldv2.pt')
    world_model.set_classes(['truck wheel', 'car wheel', 'vehicle wheel', 'tire', 'rim'])

    DATASET_DIR = '/content/dataset_ruedas_pose'
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    os.makedirs(f'{DATASET_DIR}/images/train', exist_ok=True)
    os.makedirs(f'{DATASET_DIR}/images/val', exist_ok=True)
    os.makedirs(f'{DATASET_DIR}/labels/train', exist_ok=True)
    os.makedirs(f'{DATASET_DIR}/labels/val', exist_ok=True)

    random.seed(42)
    shuffled = todas_fotos.copy()
    random.shuffle(shuffled)
    split_idx = int(len(shuffled) * 0.8) if len(shuffled) > 5 else len(shuffled)
    train_set = set(shuffled[:split_idx])

    muestras_visuales = []

    for p_img in todas_fotos:
        split = 'train' if p_img in train_set else 'val'
        fname = os.path.basename(p_img)
        stem = os.path.splitext(fname)[0]
        
        img = cv2.imread(p_img)
        if img is None:
            continue
        h, w = img.shape[:2]
        
        preds = world_model(img, conf=0.18, verbose=False)[0]
        lineas_label = []
        ruedas_vis = []
        
        if preds.boxes is not None and len(preds.boxes) > 0:
            for b in preds.boxes:
                xywhn = b.xywhn[0].cpu().numpy()
                cx, cy, bw, bh = float(xywhn[0]), float(xywhn[1]), float(xywhn[2]), float(xywhn[3])
                
                aspect = bw / max(1e-4, bh)
                if aspect < 0.35 or aspect > 2.8:
                    continue
                
                # Formato YOLO Pose: class cx cy w h kpt_x kpt_y kpt_vis
                lineas_label.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {cx:.6f} {cy:.6f} 2')
                ruedas_vis.append((int(cx * w), int(cy * h)))
                
        dst_img = f'{DATASET_DIR}/images/{split}/{fname}'
        dst_lbl = f'{DATASET_DIR}/labels/{split}/{stem}.txt'
        shutil.copy(p_img, dst_img)
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lineas_label) + ('\n' if lineas_label else ''))
            
        if len(muestras_visuales) < 3 and len(ruedas_vis) > 0:
            vis_img = img.copy()
            for rx, ry in ruedas_vis:
                cv2.circle(vis_img, (rx, ry), 10, (0, 255, 255), -1)
                cv2.circle(vis_img, (rx, ry), 12, (0, 0, 255), 2)
            muestras_visuales.append((fname, vis_img))

    yaml_path = f'{DATASET_DIR}/data.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f'path: {DATASET_DIR}\ntrain: images/train\nval: images/val\n\nkpt_shape: [1, 3]\nflip_idx: [0]\n\nnames:\n  0: rueda\n')

    shutil.make_archive('/content/dataset_ruedas_pose', 'zip', DATASET_DIR)
    print(f'✅ Auto-etiquetado finalizado. Total: {len(todas_fotos)} imágenes procesadas.')
    print(f'📦 Archivo ZIP generado (por si quieres descargarlo o subirlo a Roboflow): /content/dataset_ruedas_pose.zip')
    print(f'📄 data.yaml listo en: {yaml_path}')

    if muestras_visuales:
        fig, axes = plt.subplots(1, len(muestras_visuales), figsize=(15, 5))
        if len(muestras_visuales) == 1: axes = [axes]
        for ax, (titulo, im) in zip(axes, muestras_visuales):
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            ax.set_title(f'Keypoints: {titulo}')
            ax.axis('off')
        plt.tight_layout()
        plt.show()


## Opción B: Descargar desde Roboflow
*(Solo si ya tienes un proyecto con dataset exportado en Roboflow)*

In [ ]:
# Solo ejecuta esta celda si NO usaste la Opción A
if 'yaml_path' not in locals() or not os.path.exists(yaml_path):
    from google.colab import userdata
    from roboflow import Roboflow
    WORKSPACE = 'tu-workspace'
    PROYECTO = 'peaje-ruedas-pose'
    VERSION = 1
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11')
    yaml_path = f'{ds.location}/data.yaml'

import yaml
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['kpt_shape'] = [1, 3]
cfg['flip_idx'] = [0]
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('data.yaml verificado:', cfg)


## Entrenamiento de YOLO11s-pose

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-pose.pt')
modelo.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project='/content/drive/MyDrive/peaje/runs',
    name='m3_pose',
)


## Evaluación y Guardado de Métricas en Google Drive

In [ ]:
import glob, json, os, shutil
import numpy as np
from ultralytics import YOLO

best_pt = '/content/drive/MyDrive/peaje/runs/m3_pose/weights/best.pt'
if not os.path.exists(best_pt):
    # Buscar el best.pt más reciente si cambió el directorio de corrida
    pts = sorted(glob.glob('/content/drive/MyDrive/peaje/runs/m3_pose*/weights/best.pt'), key=os.path.getmtime)
    if pts:
        best_pt = pts[-1]

print(f'Usando pesos: {best_pt}')
mejor = YOLO(best_pt)
metricas = mejor.val(data=yaml_path)

# Obtener imágenes de validación según el origen del dataset
if 'DATASET_DIR' in locals() and os.path.exists(f'{DATASET_DIR}/images/val'):
    imgs_val = sorted(glob.glob(f'{DATASET_DIR}/images/val/*'))
    lbl_dir = f'{DATASET_DIR}/labels/val'
elif 'ds' in locals():
    imgs_val = sorted(glob.glob(f'{ds.location}/valid/images/*'))
    lbl_dir = f'{ds.location}/valid/labels'
else:
    imgs_val = []
    lbl_dir = ''

errores = []
exactos = 0
for img_p in imgs_val:
    stem = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(lbl_dir, f'{stem}.txt')
    gt = sum(1 for l in open(lbl_p, encoding='utf-8') if l.strip()) if os.path.exists(lbl_p) else 0
    pred_r = mejor(img_p, verbose=False, conf=0.35)[0]
    pred_cnt = len(pred_r.keypoints.xy[0]) if (pred_r.keypoints is not None and len(pred_r.keypoints.xy) > 0) else (len(pred_r.boxes) if pred_r.boxes is not None else 0)
    errores.append(abs(gt - pred_cnt))
    if gt == pred_cnt:
        exactos += 1

resumen = {
    'pose_map50': float(metricas.pose.map50) if hasattr(metricas, 'pose') and metricas.pose else 0.0,
    'box_map50': float(metricas.box.map50) if hasattr(metricas, 'box') and metricas.box else 0.0,
    'imagenes_val': len(imgs_val),
    'mae_ejes': float(np.mean(errores)) if errores else 0.0,
    'exactitud_conteo_ejes': float(exactos / len(imgs_val)) if imgs_val else 1.0,
}
print(json.dumps(resumen, indent=2))

with open('/content/drive/MyDrive/peaje/evidencias/metricas_m3.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2)
shutil.copy(best_pt, '/content/drive/MyDrive/peaje/modelos/m3_pose.pt')
print('✅ Modelo guardado exitosamente en Drive: /content/drive/MyDrive/peaje/modelos/m3_pose.pt')
